In [ ]:
import os
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, models, datasets

# 1. 환경 설정 및 경로
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 40,
    'WARMUP_EPOCHS': 5,
    'BATCH_SIZE': 128,
    'LR': 9e-5,
    'WEIGHT_DECAY': 0.1,
    'MIXUP_ALPHA': 1.0,
    'SEED': 42,
    'NUM_CLASSES': 4,
    'SAVE_DIR': './models',
    'BASE_PATH': r'/workspace/shared/data/processed/data_processed_v6'
}

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path(CFG['SAVE_DIR']).mkdir(exist_ok=True)

# 2. 데이터 로더
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomChoice([
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.RandomGrayscale(p=0.1),
    ]),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(CFG['BASE_PATH'], 'train'), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(CFG['BASE_PATH'], 'val'), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False, num_workers=8, pin_memory=True)

# 3. Mixup 기법
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# 4. 모델 설계
class ConvNeXtEmotionModel(nn.Module):
    def __init__(self, num_classes=4):
        super(ConvNeXtEmotionModel, self).__init__()
        self.backbone = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        in_features = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

# 5. Warmup Cosine 스케줄러
class WarmupCosineSchedule(optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_steps, t_total, last_epoch=-1):
        self.warmup_steps = warmup_steps
        self.t_total = t_total
        super(WarmupCosineSchedule, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self.last_epoch
        if step < self.warmup_steps:
            return [base_lr * float(step) / float(max(1, self.warmup_steps)) for base_lr in self.base_lrs]
        progress = float(step - self.warmup_steps) / float(max(1, self.t_total - self.warmup_steps))
        return [base_lr * 0.5 * (1.0 + np.cos(np.pi * progress)) for base_lr in self.base_lrs]

# 6. 학습 엔진 (Train/Val Acc 분리 표기)
def train(model, train_loader, val_loader, optimizer, scheduler, criterion):
    best_acc = 0
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(1, CFG['EPOCHS'] + 1):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch [{epoch}/{CFG['EPOCHS']}]")
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            
            # Mixup 적용
            mixed_imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=CFG['MIXUP_ALPHA'])
            
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(mixed_imgs)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            
            # Train Accuracy 계산 (Mixup이 적용되지 않은 원본 레이블 기준 정확도 확인)
            with torch.no_grad():
                model.eval() # 정확도 측정을 위해 잠시 eval 모드
                raw_outputs = model(imgs)
                preds = raw_outputs.argmax(1)
                train_correct += (preds == labels).sum().item()
                train_total += labels.size(0)
                model.train() # 다시 학습 모드
            
            pbar.set_postfix({
                'Loss': f"{train_loss/(pbar.n+1):.4f}",
                'T-Acc': f"{(train_correct/train_total)*100:.2f}%"
            })
            
        # 검증 단계
        val_acc = validation(model, val_loader)
        train_final_acc = (train_correct / train_total) * 100
        
        print(f"== Epoch {epoch} Summary ==")
        print(f"   Train Acc : {train_final_acc:.2f}% | Train Loss : {train_loss/len(train_loader):.4f}")
        print(f"   Val Acc   : {val_acc:.2f}%")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), os.path.join(CFG['SAVE_DIR'], 'Best_convnext7.pth'))
            print(f"   ⭐ Best Model Updated!")
            
        scheduler.step()

def validation(model, val_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return (correct / total) * 100

if __name__ == "__main__":
    model = ConvNeXtEmotionModel(num_classes=CFG['NUM_CLASSES']).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=CFG['WEIGHT_DECAY'])
    scheduler = WarmupCosineSchedule(optimizer, warmup_steps=CFG['WARMUP_EPOCHS'], t_total=CFG['EPOCHS'])
    
    print(f"🚀 학습 시작 (Target LR: {CFG['LR']})")
    train(model, train_loader, val_loader, optimizer, scheduler, criterion)